# Aprendizagem de Máquina II — RBF com Keras e TensorFlow
**Aluno: Ariel Ladislau Reises**

Rede RBF para classificar o dataset `make_moons`, com 300 amostras e ruído 0,20. **O único uso do sklearn é gerar o dataset.** A divisão estratificada e as métricas são feitas com NumPy, e a normalização e o modelo com TensorFlow/Keras.

No Google Colab, selecione **Ambiente de execução → Executar tudo**. Não é necessário usar GPU. Se seu ambiente não tiver as dependências, execute a célula opcional abaixo.

A camada oculta calcula $\phi_j(x)=\exp(-\beta_j\|x-c_j\|^2)$. Os centros começam em amostras do treino e são ajustados pelo Adam, junto com as larguras e os pesos da saída sigmoide. Usamos $\beta_j=\operatorname{softplus}(b_j)+\epsilon$ para garantir valores positivos. Esta é uma camada RBF explícita, implementada como uma camada personalizada do Keras.

Referência: [camadas personalizadas no TensorFlow/Keras](https://www.tensorflow.org/guide/keras/making_new_layers_and_models_via_subclassing).

In [ ]:
# Opcional, apenas se as bibliotecas estiverem ausentes:
# %pip install tensorflow scikit-learn numpy matplotlib

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from sklearn.datasets import make_circles  # único uso do sklearn

keras.utils.set_random_seed(42)
plt.rcParams.update({'figure.figsize': (7, 5), 'font.size': 11})
print('TensorFlow:', tf.__version__)
X, y = make_circles(n_samples=300, noise=0.20, random_state=42)
X = X.astype('float32')
y = y.astype('int32')
plt.scatter(X[:, 0], X[:, 1], c=y, cmap='plasma', edgecolors='white')
plt.title('Make Circles')
plt.xlabel('Característica 1'); plt.ylabel('Característica 2')
plt.show()

## Divisão estratificada sem sklearn
Reservamos 25% para teste (75 amostras). Do restante, 20% vão para validação (45 amostras); ficam 180 para treino. A validação orienta o EarlyStopping. O teste é usado somente na avaliação final. A divisão é reproduzível, mas não é idêntica à divisão interna do `train_test_split` do notebook anterior.

In [ ]:
def dividir_estratificado(X, y, proporcao, seed=42):
    rng = np.random.default_rng(seed)
    grupos = [rng.permutation(np.flatnonzero(y == classe)) for classe in np.unique(y)]
    total = int(round(len(y) * proporcao))
    cotas = np.array([len(g) * proporcao for g in grupos])
    quantidades = np.floor(cotas).astype(int)
    for i in np.argsort(-(cotas - quantidades))[:total - quantidades.sum()]:
        quantidades[i] += 1
    reservado = np.concatenate([g[:k] for g, k in zip(grupos, quantidades)])
    restante = np.concatenate([g[k:] for g, k in zip(grupos, quantidades)])
    rng.shuffle(reservado); rng.shuffle(restante)
    return X[restante], X[reservado], y[restante], y[reservado]

X_dev, X_teste, y_dev, y_teste = dividir_estratificado(X, y, 0.25)
X_treino, X_val, y_treino, y_val = dividir_estratificado(X_dev, y_dev, 0.20, seed=43)
for nome, rotulos in [('Treino', y_treino), ('Validação', y_val), ('Teste', y_teste)]:
    print(f'{nome}: {len(rotulos)} amostras | classes: {np.bincount(rotulos)}')
normalizacao = keras.layers.Normalization(axis=-1, name='normalizacao')
normalizacao.adapt(X_treino)
X_treino_norm = normalizacao(X_treino).numpy()

## Camada RBF personalizada
`build()` cria os parâmetros treináveis e `call()` calcula as distâncias e ativações gaussianas. Os centros e betas serão aprendidos por retropropagação.

In [ ]:
@keras.utils.register_keras_serializable(package='Ariel')
class CamadaRBF(keras.layers.Layer):
    def __init__(self, unidades=20, beta_inicial=1.0, **kwargs):
        super().__init__(**kwargs)
        self.unidades = unidades
        self.beta_inicial = beta_inicial

    def build(self, input_shape):
        self.centros = self.add_weight(
            name='centros', shape=(self.unidades, int(input_shape[-1])),
            initializer='zeros', trainable=True)
        self.beta_bruto = self.add_weight(
            name='beta_bruto', shape=(self.unidades,),
            initializer=keras.initializers.Constant(np.log(np.expm1(self.beta_inicial))),
            trainable=True)
        super().build(input_shape)

    def call(self, inputs):
        distancias2 = tf.reduce_sum(
            tf.square(tf.expand_dims(inputs, axis=1) - self.centros), axis=-1)
        betas = tf.nn.softplus(self.beta_bruto) + 1e-6
        return tf.exp(-betas * distancias2)

    def get_config(self):
        config = super().get_config()
        config.update({'unidades': self.unidades, 'beta_inicial': self.beta_inicial})
        return config

N_CENTROS = 20
entrada = keras.Input(shape=(2,), name='caracteristicas')
rbf = CamadaRBF(unidades=N_CENTROS, name='rbf')
ativacoes = rbf(normalizacao(entrada))
saida = keras.layers.Dense(1, activation='sigmoid', name='saida')(ativacoes)
modelo = keras.Model(entrada, saida, name='Rede_RBF')
rng = np.random.default_rng(42)
indices = rng.choice(len(X_treino_norm), size=N_CENTROS, replace=False)
centros_iniciais = X_treino_norm[indices].copy()
rbf.centros.assign(centros_iniciais)
modelo.compile(optimizer=keras.optimizers.Adam(learning_rate=0.01),
               loss='binary_crossentropy', metrics=['accuracy'])
modelo.summary()

## Treinamento
O EarlyStopping restaura os pesos da época com menor perda de validação. As gaussianas, os centros e a saída são treinados conjuntamente.

In [ ]:
# Limita os threads do pipeline para funcionar também em ambientes modestos.
opcoes = tf.data.Options()
opcoes.threading.private_threadpool_size = 1
opcoes.threading.max_intra_op_parallelism = 1

def dataset(X, y, treino=False):
    ds = tf.data.Dataset.from_tensor_slices((X, y.astype('float32').reshape(-1, 1)))
    if treino:
        ds = ds.shuffle(len(y), seed=42, reshuffle_each_iteration=True)
    return ds.batch(32).with_options(opcoes)

historico = modelo.fit(
    dataset(X_treino, y_treino, treino=True),
    validation_data=dataset(X_val, y_val), epochs=400, verbose=0,
    callbacks=[keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=40, min_delta=0.0001, restore_best_weights=True)])
print(f'Épocas executadas: {len(historico.history["loss"])}')
print(f'Época com menor val_loss registrada: {np.argmin(historico.history["val_loss"]) + 1}')
print('Pesos restaurados pelo EarlyStopping (min_delta=0,0001).')
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, metrica, titulo in zip(axes, ['loss', 'accuracy'], ['Entropia cruzada', 'Acurácia']):
    ax.plot(np.arange(1, len(historico.history[metrica]) + 1), historico.history[metrica], label='Treino')
    ax.plot(np.arange(1, len(historico.history[metrica]) + 1), historico.history['val_' + metrica], label='Validação')
    ax.set_title(titulo); ax.set_xlabel('Época'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Avaliação final sem sklearn
A matriz tem classes reais nas linhas e previstas nas colunas. Precisão, recall e F1 são calculados por classe.

In [ ]:
perda, acuracia = modelo.evaluate(dataset(X_teste, y_teste), verbose=0)
probabilidades = modelo(X_teste, training=False).numpy().ravel()
previsoes = (probabilidades >= 0.5).astype(int)
print(f'Acurácia no teste: {acuracia:.2%} | Perda: {perda:.4f}')
matriz = np.zeros((2, 2), dtype=int)
np.add.at(matriz, (y_teste, previsoes), 1)
print('Classe | Precisão | Recall | F1 | Suporte')
for classe in range(2):
    tp = matriz[classe, classe]
    precisao = tp / max(matriz[:, classe].sum(), 1)
    recall = tp / max(matriz[classe, :].sum(), 1)
    f1 = 2 * precisao * recall / max(precisao + recall, 1e-12)
    print(f'{classe:6} | {precisao:8.3f} | {recall:6.3f} | {f1:.3f} | {matriz[classe].sum():7}')
plt.imshow(matriz, cmap='Blues')
for i in range(2):
    for j in range(2):
        plt.text(j, i, str(matriz[i, j]), ha='center', va='center',
                 color='white' if matriz[i, j] > matriz.max() / 2 else 'black')
plt.xticks([0, 1], ['Classe 0', 'Classe 1']); plt.yticks([0, 1], ['Classe 0', 'Classe 1'])
plt.xlabel('Classe prevista'); plt.ylabel('Classe real')
plt.title('Matriz de confusão — teste'); plt.colorbar(); plt.show()

## Fronteira de decisão e centros aprendidos

In [ ]:
xx, yy = np.meshgrid(np.linspace(X[:, 0].min()-0.5, X[:, 0].max()+0.5, 180),
                     np.linspace(X[:, 1].min()-0.5, X[:, 1].max()+0.5, 180))
grade = np.column_stack([xx.ravel(), yy.ravel()]).astype('float32')
# Chamadas em blocos evitam criar um tensor muito grande para a camada RBF.
probs = np.concatenate([modelo(bloco, training=False).numpy().ravel()
                        for bloco in np.array_split(grade, 20)]).reshape(xx.shape)
media = normalizacao.mean.numpy().reshape(-1)
desvio = np.sqrt(normalizacao.variance.numpy().reshape(-1))
centros_finais = rbf.centros.numpy() * desvio + media
plt.contourf(xx, yy, probs, levels=np.linspace(0, 1, 21), cmap='plasma', alpha=0.5)
plt.contour(xx, yy, probs, levels=[0.5], colors='black')
plt.scatter(X_teste[:, 0], X_teste[:, 1], c=y_teste, cmap='plasma', edgecolors='black', label='Teste')
plt.scatter(centros_finais[:, 0], centros_finais[:, 1], c='limegreen', marker='X',
            s=90, edgecolors='black', label='Centros aprendidos')
plt.xlabel('Característica 1'); plt.ylabel('Característica 2')
plt.title('Rede RBF — fronteira e centros'); plt.legend(); plt.show()
print('Betas aprendidos:', np.round(tf.nn.softplus(rbf.beta_bruto).numpy() + 1e-6, 3))
print('Centros:', rbf.centros.shape)
for peso in modelo.trainable_weights:
    print(peso.name, tuple(peso.shape))

## Leitura dos resultados
- A combinação das gaussianas permite uma fronteira não linear para as duas luas.
- Beta maior significa uma resposta mais estreita em torno do centro.
- A normalização aprende estatísticas apenas do treino; os centros começam apenas em pontos do treino.
- Treino e validação ajudam a observar sobreajuste; a acurácia do teste deve ser analisada junto com a matriz de confusão.
- Resultados podem variar com a versão do TensorFlow e o dispositivo. Não há garantia de uma porcentagem específica.
- Esta versão tem divisão e treinamento diferentes da versão anterior; as acurácias não constituem uma comparação controlada entre implementações.

Para experimentar, altere `N_CENTROS`, `beta_inicial` ou a taxa de aprendizado e avalie pela validação. Preserve o teste para a avaliação final.